# WTI Crude Oil Price Forecasting — Stateless Methods: Systematic Backtest (Notebook 4 of 7)

This notebook simulates a rigorous production forecasting workflow:

1. Run a **rolling weekly backtest across 2025** using
   `energy_oil_backtest.yaml` for all candidate predictors.
2. Compute metrics — **CRPS** for 5/10/21-day trajectories.
3. Select the **top contender configurations** based solely on 2025
   historical performance (no peeking at 2026).
4. Let the contenders compete in the **2026 Protected Arena**
   (`energy_oil_eval.yaml`) across the geopolitical price shock and its
   aftermath — measuring adaptive real-time responsiveness and calibration.
   The eval window runs through the most recent origin whose 21-business-day
   horizon still resolves against cached data (see `scripts/fetch_wti.py`).

The line-up spans three families behind one `Predictor` interface: **baselines**
(Naive, AutoARIMA), **numerical ML** (LightGBM ± a leak-safe covariate panel),
and **LLM/agent** methods (LLM-process forecasters and a news-reading analyst
agent) — the last run on *both* project models, `gemini-3.1-flash-lite-preview`
and `gemini-3.5-flash`. Every predictor is one toggle line in the registry in
Section 2. Agent configs come from `energy_oil_forecasting.analyst_agent`.

---
## 1. Setup, Data Registration & Spec Loading

In [1]:
import warnings
from pathlib import Path

from datetime import datetime
import energy_oil_forecasting
import pandas as pd
import yaml
from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    cached_multi_backtest,
    describe_spec,
)
from aieng.forecasting.models import ADVANCED_MODEL, LITE_MODEL
from energy_oil_forecasting.data import (
    DEFAULT_WTI_COVARIATE_SERIES_IDS,
    build_wti_multivariate_service,
)

from energy_oil_forecasting.data import (
    DEFAULT_WTI_COVARIATE_SERIES_IDS,
    WTI_SERIES_ID,
    build_wti_multivariate_service,
)

warnings.filterwarnings("ignore")

# ── Mode ──────────────────────────────────────────────────────────────────────
# Set SMOKE_TEST = True to run a 2-origin, 1-sample version of the notebook
# for fast local development and end-to-end CI testing. The full specs run
# 51 backtest + 8 eval origins; smoke runs 2 + 2.
SMOKE_TEST = True 

# ── Models ────────────────────────────────────────────────────────────────────
# The project standardises on two Vector-proxy models. Every LLM and agent
# predictor below is run once per model so we can compare them head-to-head.
# (bare proxy names — no "gemini/" prefix)
MODELS = [LITE_MODEL, ADVANCED_MODEL]  # "gemini-3.1-flash-lite-preview", "gemini-3.5-flash"

# ── Derived settings (do not edit below) ─────────────────────────────────────
N_SAMPLES = 1 if SMOKE_TEST else 3  # trajectories per LLMP-Sampled call

# LightGBM hyperparameters (shared by the univariate and +covariate variants).
LAGS = 21  # one trading month of lagged target/covariate history
NUM_SAMPLES_LGBM = 100 if SMOKE_TEST else 200  # Monte-Carlo draws for quantiles
LGBM_KWARGS = {"num_threads": 1, "n_jobs": 1, "verbosity": -1}  # deterministic, quiet

# Data service: WTI target + a leak-safe covariate panel (all Yahoo Finance —
# Brent, natural gas, gasoline, gold, USD index, the USL/USO futures-curve
# contango proxy, and VIX). Non-covariate predictors simply ignore the extras,
# so one service feeds the whole leaderboard. Unavailable tickers are skipped
# with a warning, so this still runs offline / under partial connectivity.
data_service = build_wti_multivariate_service()
COVARIATES = [c for c in DEFAULT_WTI_COVARIATE_SERIES_IDS if c in set(data_service.series_ids)]

spec_dir = Path(energy_oil_forecasting.__file__).parent / "specs"
if SMOKE_TEST:
    backtest_file, eval_file = "energy_oil_smoke.yaml", "energy_oil_eval_smoke.yaml"
else:
    backtest_file, eval_file = "energy_oil_backtest.yaml", "energy_oil_eval.yaml"

with open(spec_dir / backtest_file) as f:
    backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))
with open(spec_dir / eval_file) as f:
    eval_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))



# ── Extend to a 10yr backtest / 2yr protected holdout, quarterly cadence ────
# In SMOKE_TEST mode, use a small recent window instead so the notebook can
# be smoke-tested quickly. spec_id gets its own namespace so this never
# collides with (or silently reuses) the biweekly/original cached results.
_suffix = "_smoke" if SMOKE_TEST else ""

full_df = data_service.get_series(WTI_SERIES_ID, as_of=datetime.now()).sort_values("timestamp")
data_start = full_df["timestamp"].min()
data_end = full_df["timestamp"].max()

holdout_start = data_end - pd.DateOffset(years=2)
holdout_end = data_end
backtest_start = data_start + (holdout_start - data_start) / 2
backtest_end = holdout_start

if SMOKE_TEST:
    backtest_spec.start = (data_end - pd.DateOffset(months=1)).strftime("%Y-%m-%d")
    backtest_spec.end = data_end.strftime("%Y-%m-%d")
    backtest_spec.stride = 5

    eval_spec.start = (data_end - pd.DateOffset(months=1)).strftime("%Y-%m-%d")
    eval_spec.end = data_end.strftime("%Y-%m-%d")
    eval_spec.stride = 5
else:
    backtest_spec.start = backtest_start.strftime("%Y-%m-%d")
    backtest_spec.end = backtest_end.strftime("%Y-%m-%d")
    backtest_spec.stride = 63

    eval_spec.start = holdout_start.strftime("%Y-%m-%d")
    eval_spec.end = holdout_end.strftime("%Y-%m-%d")
    eval_spec.stride = 63

backtest_spec.tasks[0].horizons = [5, 10, 21, 63]
backtest_spec.spec_id = f"energy_oil_backtest_10yr_quarterly{_suffix}"

eval_spec.tasks[0].horizons = [5, 10, 21, 63]
eval_spec.spec_id = f"energy_oil_eval_10yr_quarterly{_suffix}"





print(f"{'⚡ SMOKE MODE' if SMOKE_TEST else '📊 FULL MODE'} — MODELS={MODELS}  N_SAMPLES={N_SAMPLES}")
print(f"Covariates registered ({len(COVARIATES)}): {', '.join(COVARIATES) or '(none)'}")
print()
print("━" * 72)
print("LOADED SPECIFICATIONS:")
print("━" * 72)
print(describe_spec(backtest_spec, data_service))
print(describe_spec(eval_spec, data_service))

⚡ SMOKE MODE — MODELS=['gemini-3.1-flash-lite-preview', 'gemini-3.5-flash']  N_SAMPLES=1
Covariates registered (7): brent_log_ret_1b_l1b, natgas_log_ret_1b_l1b, gasoline_log_ret_1b_l1b, gold_log_ret_1b_l1b, dollar_index_log_ret_1b_l1b, oil_curve_contango_l1b, vix_level_l1b

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LOADED SPECIFICATIONS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MultiTargetBacktestSpec (spec_id=energy_oil_backtest_10yr_quarterly_smoke)
  description: Two-origin smoke backtest for local and CI testing of the NB04 pipeline. Uses the same tasks, horizons, and warmup as energy_oil_backtest but with only 2 weekly origins so the full notebook can be exercised without burning tokens on 51 × 5 predictor evaluations.
  start:       2026-06-23
  end:         2026-07-23
  stride:      5
  warmup:      250
  tasks:       1

Task: wti_oil_price_forecast
  description: WTI Crude Oil continuous front-month futures Close pr

---
## 2. Candidate Predictors

This experiment puts a full slate of methods on the same `Predictor` interface and
the same rolling backtest, spanning three families:

| Family | Predictors | Role |
|---|---|---|
| **Baselines** | `Naive (Last Value)`, `AutoARIMA` | Carry-forward floor + the classical statistical anchor |
| **Numerical ML** | `LightGBM`, `LightGBM + cov` (+ optional `Prophet`) | Gradient-boosted quantile regression on lagged price (and a leak-safe covariate panel — Brent, gas, gasoline, gold, USD index, the futures-curve contango proxy, and VIX). LightGBM-with-covariates was the strongest method in the S&P 500 study. |
| **LLM / Agent** | `LLMP-Sampled`, `LLMP-Grid`, `News Agent` — each on **both** project models | LLM-process forecasters and a news-reading analyst agent, run on `gemini-3.1-flash-lite-preview` *and* `gemini-3.5-flash` |

The predictor cell below is a **registry**: every method is one line with an
`enabled` flag. Flip a flag to add or drop a predictor — the rest of the
notebook (backtest, scoring, eval, scorecard) iterates over whatever is active.
The two baselines are flagged `baseline=True` and are the only results written to
`adaptive_agent/curriculum/` for Notebooks 5–6, so toggling the others never
disturbs the downstream training data.

In [2]:
from dataclasses import dataclass
from typing import Callable

from aieng.forecasting.methods import (
    LastValuePredictor,
    QuantileGridLLMPredictor,
    QuantileGridLLMPredictorConfig,
    SampledTrajectoryLLMPredictor,
    SampledTrajectoryLLMPredictorConfig,
)
from aieng.forecasting.methods.numerical.darts_arima import DartsAutoARIMAPredictor
from aieng.forecasting.methods.numerical.darts_regression import DartsLightGBMPredictor
from energy_oil_forecasting.analyst_agent import (
    build_wti_agent_predictor,
    build_wti_news_config,
    build_wti_news_contrarian_config,
    build_wti_news_factors_v2_config,
    build_wti_news_scenario_schema_config,
    build_wti_scenario_schema_predictor,
)
from energy_oil_forecasting.prophet_baseline import ProphetPredictor


@dataclass
class PredictorEntry:
    """One row in the experiment. Flip ``enabled`` to switch a predictor on/off."""

    name: str
    factory: Callable[[], object]  # lazy — built only when enabled
    enabled: bool = True
    baseline: bool = False  # baselines are saved to curriculum/ for NB05–06


# LLM / agent factories — each takes a model so the same recipe runs on both.
# LLMP-Sampled optionally serializes the covariate panel into the prompt
# (labeled exogenous-series blocks); the others are target-only. A distinct
# variant_tag keeps the +cov run separate in the cache and on the leaderboard.
def _llmp_sampled(model, covariates=None):
    return SampledTrajectoryLLMPredictor(
        SampledTrajectoryLLMPredictorConfig(
            model=model,
            n_samples=N_SAMPLES,
            covariate_series_ids=covariates,
            variant_tag="cov" if covariates else None,
        )
    )


def _llmp_grid(model):
    return QuantileGridLLMPredictor(QuantileGridLLMPredictorConfig(model=model))


#def _news_agent(model):
#    return build_wti_agent_predictor(build_wti_news_config(model=model))


#def _news_agent_contrarian(model):
#    return build_wti_agent_predictor(build_wti_news_contrarian_config(model=model))


def _news_agent(model):
    return build_wti_agent_predictor(
        build_wti_news_config(model=model, verifier_max_attempts=1, verifier_confidence_threshold=6
                              )
    )


def _news_agent_contrarian(model):
    return build_wti_agent_predictor(
        build_wti_news_contrarian_config(model=model, verifier_max_attempts=1, verifier_confidence_threshold=6
                                         )
    )


def _news_agent_factors_v2(model):
    return build_wti_agent_predictor(
        build_wti_news_factors_v2_config(model=model, verifier_max_attempts=1, verifier_confidence_threshold=6
                                         )
    )

def _news_agent_scenario_schema(model):
    return build_wti_scenario_schema_predictor(
        build_wti_news_scenario_schema_config(model=model)
    )


# ── Experiment registry ───────────────────────────────────────────────────────
# Toggle `enabled` on any line to include/exclude that predictor. LLM and agent
# methods are listed once per model so each can be switched on/off individually.
from aieng.forecasting.methods.numerical.darts_classical import DartsKalmanForecasterPredictor

REGISTRY = [
    PredictorEntry("Naive (Last Value)", LastValuePredictor, enabled=True, baseline=True),
    PredictorEntry("Kalman", DartsKalmanForecasterPredictor, enabled=True, baseline=True),
    PredictorEntry(
        "LightGBM + cov",
        lambda: DartsLightGBMPredictor(
            lags=LAGS, lags_past_covariates=LAGS, covariate_series_ids=COVARIATES,
            num_samples=NUM_SAMPLES_LGBM, lgbm_kwargs=LGBM_KWARGS,
        ),
        enabled=False,
    ),
    PredictorEntry(f"News Agent ({LITE_MODEL})", lambda: _news_agent(LITE_MODEL), enabled=True),
    PredictorEntry(
        f"News Agent Scenario Schema ({LITE_MODEL})",
        lambda: build_wti_scenario_schema_predictor(build_wti_news_scenario_schema_config(model=LITE_MODEL)),
        enabled=True,
    ),
]

# Instantiate only the enabled predictors (lazy factories skip the rest).
PREDICTORS = {e.name: e.factory() for e in REGISTRY if e.enabled}
_BASELINE_PREDICTORS = {e.name for e in REGISTRY if e.baseline}

print(f"Active predictors ({len(PREDICTORS)}):")
for name in PREDICTORS:
    tag = "  (baseline → curriculum/)" if name in _BASELINE_PREDICTORS else ""
    print(f"  {name}{tag}")

Active predictors (4):
  Naive (Last Value)  (baseline → curriculum/)
  Kalman  (baseline → curriculum/)
  News Agent (gemini-3.1-flash-lite-preview)
  News Agent Scenario Schema (gemini-3.1-flash-lite-preview)


---
## 3. Run the 2025 Historical Backtest

All 51 weekly origins in 2025 are evaluated for each predictor.
`cached_multi_backtest` caches results under `data/predictions/` so
subsequent runs are instant.

In [3]:
import time

print(f"Running rolling backtest ({backtest_spec.start} → {backtest_spec.end}, {len(PREDICTORS)} predictor(s))...")
print("LLM/agent runs are expensive — first run will take several minutes.\n")

backtest_results: dict[str, object] = {}
for i, (_name, _predictor) in enumerate(PREDICTORS.items()):
    if i > 0:
        time.sleep(15)  # pace between predictors, not just after a failure
    backtest_results[_name] = cached_multi_backtest(
        _predictor, backtest_spec, data_service,
        max_retries=4,     # was 2
        retry_delay=30.0,  # was 2.0 — long enough to clear an RPM window
        force_refresh=(_name.startswith("News Agent Scenario Schema")),
    )
    print(f"  {_name} ✓")

print(f"\nBacktest complete ({backtest_spec.start} → {backtest_spec.end}).")

Running rolling backtest (2026-06-23 → 2026-07-23, 4 predictor(s))...
LLM/agent runs are expensive — first run will take several minutes.

  Naive (Last Value) ✓
  Kalman ✓
  News Agent (gemini-3.1-flash-lite-preview) ✓


Raw agent response (schema validation failed):
{
  "forecasts": [
    {
      "horizon": 5,
      "point_forecast": 73.5,
      "quantiles": [
        { "quantile": 0.05, "value": 69.5 },
        { "quantile": 0.1, "value": 70.5 },
        { "quantile": 0.2, "value": 71.8 },
        { "quantile": 0.3, "value": 72.6 },
        { "quantile": 0.4, "value": 73.1 },
        { "quantile": 0.5, "value": 73.5 },
        { "quantile": 0.6, "value": 74.0 },
        { "quantile": 0.7, "value": 74.8 },
        { "quantile": 0.8, "value": 75.9 },
        { "quantile": 0.9, "value": 77.5 },
        { "quantile": 0.95, "value": 78.8 }
      ],
      "rationale": "Following the sharp correction from earlier peaks, near-term price action is dominated by bearish sentiment as the war premium evaporates. Expect continued downward momentum and short-term volatility as traders adjust to the normalization of supply."
    },
    {
      "horizon": 10,
      "point_forecast": 72.0,
      "quantiles": [
       

  News Agent Scenario Schema (gemini-3.1-flash-lite-preview) ✓

Backtest complete (2026-06-23 → 2026-07-23).


---
## 4. Performance Characterisation

We score every active predictor on the 2025 backtest data:
- **CRPS** (Continuous Ranked Probability Score) — sharpness + calibration combined
- **MAE at h=21d** — point forecast accuracy at the longest horizon

The leaderboard ranks the families against each other — how much structure the
numerical methods (AutoARIMA, LightGBM ± covariates) extract over the naive
floor, whether the covariate panel earns its keep, and how the LLM/agent methods
compare across the two models. Where each method wins and where it struggles in
2025 is exactly the material the adaptive agent learns from in Notebook 5.

In [4]:
import math

import numpy as np
from aieng.forecasting.evaluation.prediction import ContinuousForecast
from energy_oil_forecasting.analysis import score_backtest_results


leaderboard_rows = []

for name, results in backtest_results.items():
    scores = score_backtest_results(results, data_service)

    # score_backtest_results' "mae_h21" is actually MAE blended across every
    # horizon in the task — its mae_horizon=21 parameter isn't wired to any
    # filtering inside it. Computing a real per-horizon breakdown here
    # instead, since horizon 63 (quarterly) is the point of this run and
    # shouldn't be silently averaged in under a "21d" label.
    horizon_errors: dict[int, list[float]] = {h: [] for h in backtest_spec.tasks[0].horizons}
    for result in results.values():
        task = result.spec.task
        offset = pd.tseries.frequencies.to_offset(task.frequency)
        actual_df = data_service.get_series(task.target_series_id, as_of=datetime.now())
        actual_by_date = {
            pd.Timestamp(row["timestamp"]).normalize(): float(row["value"]) for _, row in actual_df.iterrows()
        }
        for pred in result.predictions:
            if not isinstance(pred.payload, ContinuousForecast):
                continue
            fd = pd.Timestamp(pred.forecast_date).normalize()
            actual = actual_by_date.get(fd)
            if actual is None:
                continue
            as_of = pd.Timestamp(pred.as_of)
            for h in task.horizons:
                if (as_of + offset * h).normalize() == fd:
                    horizon_errors[h].append(abs(pred.payload.point_forecast - actual))
                    break

    row = {
        "Predictor": name,
        "Mean CRPS": scores.get("mean_crps", float("nan")),
        "80% CI Coverage": scores.get("coverage_80", float("nan")),
    }
    for h in backtest_spec.tasks[0].horizons:
        errs = horizon_errors[h]
        row[f"MAE h={h}d"] = float(np.mean(errs)) if errs else float("nan")
    leaderboard_rows.append(row)

df_leaderboard = pd.DataFrame(leaderboard_rows).set_index("Predictor")
df_leaderboard = df_leaderboard.sort_values("Mean CRPS")

print("━" * 72)
print(f"BACKTEST PERFORMANCE SUMMARY ({backtest_spec.start} → {backtest_spec.end}):")
print("━" * 72)
print(df_leaderboard.to_string())

kalman_crps = df_leaderboard.loc["Kalman", "Mean CRPS"] if "Kalman" in df_leaderboard.index else float("nan")
naive_crps = (
    df_leaderboard.loc["Naive (Last Value)", "Mean CRPS"]
    if "Naive (Last Value)" in df_leaderboard.index
    else float("nan")
)
if not math.isnan(kalman_crps) and not math.isnan(naive_crps):
    print(
        f"\nKalman CRPS improvement over Naive: {naive_crps - kalman_crps:.4f} "
        f"({(naive_crps - kalman_crps) / naive_crps:.1%})"
    )

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BACKTEST PERFORMANCE SUMMARY (2026-06-23 → 2026-07-23):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                                            Mean CRPS  80% CI Coverage  MAE h=5d  MAE h=10d  MAE h=21d  MAE h=63d
Predictor                                                                                                                        
News Agent (gemini-3.1-flash-lite-preview)                   6.206023             12.5  5.797501   9.936666  15.330002        NaN
News Agent Scenario Schema (gemini-3.1-flash-lite-preview)   6.929103             12.5  6.047501  10.503333  15.330002        NaN
Kalman                                                       7.399877             12.5  6.054837   9.915505  12.092063        NaN
Naive (Last Value)                                           8.066249              0.0  5.797499   9.776665  12.010002        NaN

Kalman CRPS impro

In [5]:
# ── Skipped: shared curriculum/ files feed NB05 ("explores 2025 data") and
# NB06 — this notebook's backtest_spec is the 10yr/quarterly window, not 2025
# weekly, so writing here would silently corrupt their training input.
#
# _CURRICULUM_DIR = Path("adaptive_agent/curriculum")
# _CURRICULUM_DIR.mkdir(exist_ok=True)
# for _name, _result_dict in backtest_results.items():
#     if _name not in _BASELINE_PREDICTORS:
#         continue
#     _result = next(iter(_result_dict.values()))
#     (_CURRICULUM_DIR / f"backtest_{_name}.json").write_text(_result.model_dump_json(), encoding="utf-8")
# print(f"Saved {sum(n in _BASELINE_PREDICTORS for n in backtest_results)} backtest result(s) to {_CURRICULUM_DIR}/")

In [6]:
from collections import Counter
from energy_oil_forecasting.analysis import _business_horizon
import pandas as pd

for name in ["Kalman", "Naive (Last Value)",
             "News Agent Scenario Schema (gemini-3.1-flash-lite-preview)",
             "News Agent (gemini-3.1-flash-lite-preview)"]:
    result = next(iter(backtest_results[name].values()))
    origins = sorted({pd.Timestamp(p.as_of).date() for p in result.predictions})
    horizons_present = Counter(
        _business_horizon(pd.Timestamp(p.as_of), pd.Timestamp(p.forecast_date)) for p in result.predictions
    )
    print(f"{name}: origins={origins}")
    print(f"  horizon counts: {dict(horizons_present)}")

Kalman: origins=[datetime.date(2026, 6, 23), datetime.date(2026, 6, 30), datetime.date(2026, 7, 7), datetime.date(2026, 7, 14)]
  horizon counts: {5: 4, 10: 3, 21: 1}
Naive (Last Value): origins=[datetime.date(2026, 6, 23), datetime.date(2026, 6, 30), datetime.date(2026, 7, 7), datetime.date(2026, 7, 14)]
  horizon counts: {5: 4, 10: 3, 21: 1}
News Agent Scenario Schema (gemini-3.1-flash-lite-preview): origins=[datetime.date(2026, 6, 23), datetime.date(2026, 6, 30), datetime.date(2026, 7, 7), datetime.date(2026, 7, 14)]
  horizon counts: {5: 4, 10: 3, 21: 1}
News Agent (gemini-3.1-flash-lite-preview): origins=[datetime.date(2026, 6, 23), datetime.date(2026, 6, 30), datetime.date(2026, 7, 7), datetime.date(2026, 7, 14)]
  horizon counts: {5: 4, 10: 3, 21: 1}


---
## 5. 2026 Evaluation — Held-Out Test Period

We run every active predictor on **18 weekly origins spanning Feb–Jun 2026**
(`energy_oil_eval.yaml`) — the major geopolitical volatility spike not seen
during the 2025 backtest, plus its aftermath. The window runs through the
most recent origin that still fully resolves against cached WTI data (the
21-business-day horizon needs data 21 business days past the origin).

This evaluation serves two purposes:
1. **Measure out-of-sample robustness** — do the 2025 edges (statistical,
   covariate, or LLM/agent) hold under a structural regime shift?
2. **Establish the stateless baseline** that the trained adaptive agents in
   Notebook 6 are compared against. The baseline predictors' results are saved
   to `adaptive_agent/curriculum/` for Notebooks 5 and 6 to load.

In [7]:
import time

print(f"Running evaluation ({eval_spec.start} → {eval_spec.end}, {len(PREDICTORS)} predictor(s))...")
eval_results: dict[str, object] = {}
for i, (name, predictor) in enumerate(PREDICTORS.items()):
    if i > 0:
        time.sleep(15)
    eval_results[name] = cached_multi_backtest(
        predictor, eval_spec, data_service,
        max_retries=4,
        retry_delay=30.0,
        force_refresh=(name.startswith("News Agent Scenario Schema")),
    )
    print(f"  {name} ✓")

print(f"\nEvaluation complete ({eval_spec.start} → {eval_spec.end}).")

Running evaluation (2026-06-23 → 2026-07-23, 4 predictor(s))...
  Naive (Last Value) ✓
  Kalman ✓
  News Agent (gemini-3.1-flash-lite-preview) ✓


Raw agent response (schema validation failed):
{
  "forecasts": [
    {
      "horizon": 5,
      "point_forecast": 82.5,
      "quantiles": [
        {
          "quantile": 0.05,
          "value": 75.0
        },
        {
          "quantile": 0.1,
          "value": 77.0
        },
        {
          "quantile": 0.2,
          "value": 79.0
        },
        {
          "quantile": 0.3,
          "value": 80.5
        },
        {
          "quantile": 0.4,
          "value": 81.5
        },
        {
          "quantile": 0.5,
          "value": 82.5
        },
        {
          "quantile": 0.6,
          "value": 83.5
        },
        {
          "quantile": 0.7,
          "value": 85.0
        },
        {
          "quantile": 0.8,
          "value": 87.5
        },
        {
          "quantile": 0.9,
          "value": 91.0
        },
        {
          "quantile": 0.95,
          "value": 95.0
        }
      ],
      "rationale": "Over a 5-day horizon, the market is

  News Agent Scenario Schema (gemini-3.1-flash-lite-preview) ✓

Evaluation complete (2026-06-23 → 2026-07-23).


In [8]:
# ── Skipped: this notebook's eval_spec is the 10yr/quarterly holdout, not
# the original 2026 shock-period window NB05/NB06 expect from these files.
# Writing here would silently overwrite their curriculum inputs.
#
# for _name, _result_dict in eval_results.items():
#     if _name not in _BASELINE_PREDICTORS:
#         continue
#     _result = next(iter(_result_dict.values()))
#     (_CURRICULUM_DIR / f"eval_{_name}.json").write_text(_result.model_dump_json(), encoding="utf-8")
# print(f"Saved {sum(n in _BASELINE_PREDICTORS for n in eval_results)} eval result(s) to {_CURRICULUM_DIR}/")

---
## 6. Scorecard

Out-of-sample performance of every active predictor on the 2026 eval period.
These numbers are the **stateless baseline** the adaptive agent variants must
beat in Notebook 6 to demonstrate that training added value.

In [9]:
import numpy as np
from aieng.forecasting.evaluation.prediction import ContinuousForecast
from energy_oil_forecasting.analysis import score_backtest_results


scorecard_rows = []
for name in PREDICTORS:
    if name not in eval_results:
        continue
    results = eval_results[name]
    scores = score_backtest_results(results, data_service)

    # Same reasoning as the backtest leaderboard: score_backtest_results'
    # "mae_h21" is blended across every horizon, not horizon-21-specific.
    horizon_errors: dict[int, list[float]] = {h: [] for h in eval_spec.tasks[0].horizons}
    for result in results.values():
        task = result.spec.task
        offset = pd.tseries.frequencies.to_offset(task.frequency)
        actual_df = data_service.get_series(task.target_series_id, as_of=datetime.now())
        actual_by_date = {
            pd.Timestamp(row["timestamp"]).normalize(): float(row["value"]) for _, row in actual_df.iterrows()
        }
        for pred in result.predictions:
            if not isinstance(pred.payload, ContinuousForecast):
                continue
            fd = pd.Timestamp(pred.forecast_date).normalize()
            actual = actual_by_date.get(fd)
            if actual is None:
                continue
            as_of = pd.Timestamp(pred.as_of)
            for h in task.horizons:
                if (as_of + offset * h).normalize() == fd:
                    horizon_errors[h].append(abs(pred.payload.point_forecast - actual))
                    break

    row = {
        "Predictor": name,
        "Mean CRPS": scores.get("mean_crps", float("nan")),
        "80% CI Coverage": scores.get("coverage_80", float("nan")),
    }
    for h in eval_spec.tasks[0].horizons:
        errs = horizon_errors[h]
        row[f"MAE h={h}d"] = float(np.mean(errs)) if errs else float("nan")
    scorecard_rows.append(row)

df_scorecard = pd.DataFrame(scorecard_rows).set_index("Predictor")
df_scorecard = df_scorecard.sort_values("Mean CRPS")

print("━" * 72)
print(f"EVAL SCORECARD ({eval_spec.start} → {eval_spec.end}):")
print("━" * 72)
print(df_scorecard.to_string())

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EVAL SCORECARD (2026-06-23 → 2026-07-23):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                                            Mean CRPS  80% CI Coverage  MAE h=5d  MAE h=10d  MAE h=21d  MAE h=63d
Predictor                                                                                                                        
News Agent (gemini-3.1-flash-lite-preview)                   6.454824             25.0  5.922501  10.603333  15.830002        NaN
News Agent Scenario Schema (gemini-3.1-flash-lite-preview)   7.070194             25.0  5.547501   9.796667  19.330002        NaN
Kalman                                                       7.385701             12.5  6.134081   9.876659  12.112239        NaN
Naive (Last Value)                                           8.066249              0.0  5.797499   9.776665  12.010002        NaN


---
## 7. Diagnostics — reading past the leaderboard

The scorecard above is a single number per method. That hides *where* the score
comes from and *whether the ranking is even real*. The next cells decompose it
straight from the eval predictions — so they recompute on any rerun, smoke or
full:

- **CRPS by horizon** — does a method win everywhere, or is its mean dominated by
  one horizon? (For a short forecast, the 5-day calls are easy and nearly tied;
  the ranking is usually decided by the longest horizon.)
- **Mean CRPS ± standard error** — with only a handful of origins, are the gaps
  between methods bigger than the noise, or is the "winner" a coin flip?

With the **smoke spec (2 origins → a few scored points)** expect wide error bars
and an unstable ranking. That is exactly why a surprising leaderboard here is not
yet evidence of anything — it is a pipeline check.

In [10]:
from energy_oil_forecasting import viz
from energy_oil_forecasting.analysis import (
    build_price_frame,
    eval_narrative_md,
    extract_agent_rationales,
    leaderboard_with_uncertainty,
    per_horizon_crps,
    predictions_to_frame,
)
from IPython.display import HTML, Markdown, display  # noqa: A004


# Explode every scored 2026 eval prediction into one tidy row per
# (predictor, origin, horizon): point, 80% interval, realised price, and CRPS.
# Everything in Sections 7–10 reads from this frame, so it all recomputes when
# you flip SMOKE_TEST off and rerun.
price_df = build_price_frame(data_service)
eval_frame = predictions_to_frame(eval_results, data_service)
eval_board = leaderboard_with_uncertainty(eval_frame)
ph_crps = per_horizon_crps(eval_frame)

print("━" * 72)
print("MEAN CRPS BY PREDICTOR × HORIZON (lower = better; 'All' = overall mean):")
print("━" * 72)
print(ph_crps.round(2).to_string())

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MEAN CRPS BY PREDICTOR × HORIZON (lower = better; 'All' = overall mean):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                                            h=5d  h=10d  h=21d   All
predictor                                                                           
News Agent (gemini-3.1-flash-lite-preview)                  4.34   8.18   9.74  6.45
News Agent Scenario Schema (gemini-3.1-flash-lite-preview)  4.12   8.18  15.57  7.07
Kalman                                                      5.44   8.80  10.95  7.39
Naive (Last Value)                                          5.80   9.78  12.01  8.07


In [11]:
# Heatmap of the table above. Read it left-to-right: the short-horizon columns
# are usually a near-uniform green (everyone is right), and one long-horizon
# column carries the colour spread that sets the 'All' ranking.
viz.make_crps_heatmap(ph_crps)

In [12]:
# Same leaderboard, now with a standard-error bar on each mean. If the bars of
# the top methods overlap, their ordering is not statistically distinguishable —
# the honest verdict when only a few origins have been scored.
viz.make_leaderboard_interval_chart(eval_board)

---
## 8. What are the top methods actually forecasting?

A CRPS number doesn't show *behaviour*. Below, each leading method's **median
forecast and 80% interval** are drawn against the realised WTI path at every
eval origin. This is where the leaderboard becomes legible — watch for who
tracks the move, who simply anchors to the last price, and whose intervals are
too narrow to cover the outcome when the market jumps.

In [13]:
# Plot the leaderboard's top methods, and always include the best LLM/agent
# method for contrast (so the chart compares families even when a baseline leads).
_leaders = list(eval_board.index[:3])
_best_llm = next((p for p in eval_board.index if eval_board.loc[p, "family"] == "LLM / Agent"), None)
if _best_llm and _best_llm not in _leaders:
    _leaders.append(_best_llm)
print(f"Showing: {', '.join(_leaders)}")
viz.make_eval_forecast_chart(eval_frame, price_df, _leaders)

Showing: News Agent (gemini-3.1-flash-lite-preview), News Agent Scenario Schema (gemini-3.1-flash-lite-preview), Kalman


---
## 9. Reading the agent's reasoning

The news-reading agent attaches a free-text **rationale** to every forecast, and
a link to the full **Langfuse trace**. These are pulled straight from the
prediction metadata. This is where a surprising score becomes interpretable: you
can read whether the agent actually saw the geopolitical risk, and *how* it
turned that into a price and an interval — including, often, an interval far too
narrow for a regime shift.

In [14]:
# One card per (agent, origin): the rationale, the per-horizon note, and a link
# to the full reasoning trace. Empty only if no LLM/agent predictor is enabled.
eval_rationales = extract_agent_rationales(eval_results)
display(HTML(viz.render_rationales_html(eval_rationales)))

---
## 10. Takeaways — computed from this run

The summary below is **generated from the eval results in memory, not
hard-coded**, so it always matches what actually ran: the real winner, whether
its lead clears the noise floor, the horizon that decided the ranking, the
best-performing family, and a calibration line. Flip `SMOKE_TEST` off, rerun,
and these takeaways update themselves with the full leaderboard.

In [15]:
display(Markdown(eval_narrative_md(eval_frame, smoke=SMOKE_TEST)))

1. **News Agent (gemini-3.1-flash-lite-preview)** has the best mean CRPS (6.45) on the 2026 evaluation, ahead of **News Agent Scenario Schema (gemini-3.1-flash-lite-preview)** (7.07) by 0.62 — **well within the combined standard error**, so the ranking here is not statistically distinguishable from noise.
2. The leaderboard is **decided at h=10d**, where CRPS ranges 1.1–16.4 across methods; at the short h=21d horizon the methods are nearly tied (range 9.7–15.6). A handful of long-horizon points drives the whole ranking.
3. **By family** (mean CRPS): LLM / Agent 6.76, Other 7.39, Baseline 8.07. Best family this window: **LLM / Agent**.
4. **Calibration:** News Agent (gemini-3.1-flash-lite-preview)'s 80% interval covered 25% of outcomes (target 80%) over its 8 scored point(s). With this few, coverage this far from target is itself a small-sample artefact, not necessarily mis-calibration.
5. ⚠️ **Smoke run:** only 4 origin(s) / 32 scored points. Treat the ranking as a pipeline check, not evidence — rerun the full suite before drawing conclusions.

---
## 11. What stateless methods can't do

Sections 7–10 score and dissect this run on its own terms. But every method here
shares one structural limit, independent of who topped the leaderboard: it is
calibrated (or prompted) **once and never updated between rounds**. That is
intentional — it creates a clean baseline — but it leaves a systematic gap:

- **No error feedback.** If a method's intervals are consistently too narrow in
  an elevated-vol regime (read the coverage line in Section 10, and the squashed
  error bars in Section 8), it keeps making the same mistake. Nothing updates its
  calibration between origins.

- **No strategy evolution.** Each prediction starts from the same prior — the
  same fitted model, or the same prompt. Resolved outcomes disappear without
  influencing future forecasts.

- **Context without memory.** Even the news agent re-reads the world each origin;
  it does not accumulate what worked. The rationales in Section 9 are written
  fresh every time, with no record of how the last one resolved.

→ **Notebook 5** introduces adaptive agents that study the 2025 backtest, record
systematic observations, and calibrate their strategies accordingly. At inference
time, each agent receives the live stateless estimate and decides how to adjust
it — applying what it learned from training.

→ **Notebook 6** evaluates whether any training approach actually improved
out-of-sample performance on the held-out 2026 data — measured against the
stateless baseline this notebook just established.